In [4]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

In [ ]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'
scenario_list = {"base":("SSP2",["base"])}
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dirs = {
        scenario: scenario_base_path / scenario for scenario in circular_scen
    }

In [6]:
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path("..", "data", "raw", "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dirs = {
        scenario: scenario_base_path / scenario for scenario in circular_scen
    }

    bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs)
    vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs)

    # TODO fix this for real in the future
    prep_data = vhc_sector.prep_data

    target_materials = [
    "Aluminium", "Brick", "Cement", "Concrete", 
    "Copper", "Glass", "Steel", "Wood"
    ]

    prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['battery_materials'] = prep_data['battery_materials'].reindex(material=target_materials)
    prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['material_fractions'] = prep_data['material_fractions'].reindex(material=target_materials)
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].reindex(material=target_materials)

    vhc_sector = Sector('vehicles', prep_data)

    prep_eol = eol_preprocess(Path("..", "data", "raw"), circular_economy_scenario_dirs)
    eol_sector = Sector(name="eol", data = prep_eol)

    factory = ModelFactory(
    [bld_sector, vhc_sector, eol_sector], complete_timeline
    ).add(GenericStocks, ["vehicles", "buildings"]
    ).add(GenericMaterials, "vehicles"
    ).add(Maintenance, "vehicles"
    ).add(MaterialIntensities, "buildings",
    ).add(EndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": ["vehicles", "buildings"],
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
)
    model = factory.finish()

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    all_output[scen_id] = {
        "inflow_materials": [model.vehicles["inflow_materials"], model.buildings["inflow_materials"]],
        "reusable_materials": model.eol["reusable_materials"],
        "recyclable_materials": model.eol["recyclable_materials"]
    }
    print(f"Finished {scen_id}")

c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1566: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppDat

unit scaling factors: dimensionless
floorspace_commercial: meter ** 2 / person
implemented 'base' for Commercial Buildings


c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)


implemented 'narrow' for Commercial Buildings
current_vals: meter ** 2 / person
implemented 'base' for Residential Buildings
unit: meter ** 2


c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1566: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


DimensionalityError: Cannot convert from 'meter ** 2 * person' ([length] ** 2 * [population]) to 'meter ** 2' ([length] ** 2)

In [7]:
bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs)

c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1566: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\numpy\lib\_function_base_impl.py:2622: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  args = tuple(asanyarray(arg) for arg in args)
c:\Users\Arp00003\AppDat

unit scaling factors: dimensionless
floorspace_commercial: meter ** 2 / person
implemented 'base' for Commercial Buildings
implemented 'narrow' for Commercial Buildings
current_vals: meter ** 2 / person
implemented 'base' for Residential Buildings
unit: meter ** 2


c:\Users\Arp00003\AppData\Local\anaconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1566: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


DimensionalityError: Cannot convert from 'meter ** 2 * person' ([length] ** 2 * [population]) to 'meter ** 2' ([length] ** 2)

In [ ]:
model.vehicles.keys()

In [ ]:
model.eol.keys()

In [ ]:
import matplotlib.pyplot as plt
model.eol.get('collected_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').plot(label = "Collected Materials")
model.eol.get('reusable_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').plot(label = "Reusable Materials")
model.eol.get('recyclable_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').plot(label = "Recyclable Materials")
model.eol.get('losses_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').plot(label = "Losses Materials")
plt.legend()


In [ ]:
model.vehicles.get('inflow_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').loc[1961: ].plot(label = "Inflow vehicles")
model.buildings.get('inflow_materials').to_array().sum(["Region", "Type"]).sel(material = "Steel").pint.to('Mt').loc[1961: ].plot(label = "Inflow buildings")
plt.legend()